# 07 - Análise de Sensibilidade
## Testando a robustez dos resultados
Variamos os principais parâmetros do pipeline, um de cada vez,
para verificar se os resultados são estáveis ou dependem de escolhas específicas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("../data/processed/features_with_regime.parquet")
prices = pd.read_parquet("../data/raw/sp500_daily_prices.parquet")

print(f"Features: {df.shape}")
print(f"Preços: {prices.shape}")

### Funções auxiliares
Encapsulamos o treinamento walk-forward e a simulação de portfólio
em funções reutilizáveis para rodar múltiplos cenários.

In [ ]:
exclude_cols = [
    "open", "high", "low", "close", "adj_close", "volume",
    "target_21d", "dollar_volume", "obv",
    "macd", "macd_signal", "market_regime_name"
]
feature_cols = [c for c in df.columns if c not in exclude_cols]
target_col = "target_21d"

dates = df.index.get_level_values("date")
unique_dates = sorted(dates.unique())
monthly_ends = pd.Series(unique_dates).groupby(
    pd.Series(unique_dates).dt.to_period("M")
).last().values
min_train_days = 504
monthly_ends_filtered = [d for d in monthly_ends if d >= unique_dates[min_train_days]]

print(f"Features: {len(feature_cols)}")
print(f"Meses de rebalanceamento: {len(monthly_ends_filtered)}")

In [ ]:
def run_walkforward(xgb_params, feature_cols=feature_cols, df=df):
    all_preds = []
    for i in range(len(monthly_ends_filtered) - 1):
        train_end = monthly_ends_filtered[i]
        test_end = monthly_ends_filtered[i + 1]

        train_mask = dates <= train_end
        test_mask = (dates > train_end) & (dates <= test_end)

        train_data = df.loc[train_mask].dropna(subset=[target_col])
        test_data = df.loc[test_mask].dropna(subset=[target_col])

        if len(train_data) < 100 or len(test_data) < 10:
            continue

        model = xgb.XGBRegressor(**xgb_params)
        model.fit(train_data[feature_cols], train_data[target_col], verbose=False)
        preds = model.predict(test_data[feature_cols])

        pred_df = pd.DataFrame({
            "date": test_data.index.get_level_values("date"),
            "ticker": test_data.index.get_level_values("ticker"),
            "predicted_return": preds,
            "actual_return": test_data[target_col].values
        })
        all_preds.append(pred_df)

    return pd.concat(all_preds, ignore_index=True)


def evaluate_predictions(predictions):
    daily_corrs = []
    for date, group in predictions.groupby("date"):
        if len(group) < 10:
            continue
        corr, _ = spearmanr(group["predicted_return"], group["actual_return"])
        daily_corrs.append(corr)

    predictions["quintile"] = predictions.groupby("date")["predicted_return"].transform(
        lambda x: pd.qcut(x, 5, labels=[1, 2, 3, 4, 5], duplicates="drop")
    ).astype(int)
    quintile_returns = predictions.groupby("quintile")["actual_return"].mean()
    spread = quintile_returns.get(5, 0) - quintile_returns.get(1, 0)

    return {
        "spearman_mean": np.mean(daily_corrs),
        "spearman_positive_pct": np.mean([c > 0 for c in daily_corrs]),
        "spread_q5_q1": spread,
        "q1_return": quintile_returns.get(1, np.nan),
        "q5_return": quintile_returns.get(5, np.nan)
    }


def simulate_portfolio(predictions, top_pct=0.2, transaction_cost=0.001):
    predictions["date"] = pd.to_datetime(predictions["date"])
    predictions["year_month"] = predictions["date"].dt.to_period("M")
    monthly_last = predictions.groupby("year_month")["date"].max()

    close_wide = prices["close"].unstack("ticker")
    close_wide.index = pd.to_datetime(close_wide.index)
    daily_ret = close_wide.pct_change()

    selections = {}
    for ym, last_day in monthly_last.items():
        day_preds = predictions[predictions["date"] == last_day].copy()
        if len(day_preds) < 20:
            continue
        day_preds["rank"] = day_preds["predicted_return"].rank(pct=True)
        top = day_preds[day_preds["rank"] >= (1 - top_pct)]["ticker"].tolist()
        if len(top) >= 5:
            selections[last_day] = top

    rebal_dates = sorted(selections.keys())
    port_returns = []
    prev_tickers = []

    for i in range(len(rebal_dates) - 1):
        start = rebal_dates[i]
        end = rebal_dates[i + 1]
        tickers = selections[start]
        n = len(tickers)
        weight = 1.0 / n

        turnover = 0.0
        all_t = set(tickers + prev_tickers)
        for t in all_t:
            old_w = weight if t in prev_tickers else 0.0
            new_w = weight if t in tickers else 0.0
            turnover += abs(new_w - old_w)
        cost = turnover * transaction_cost

        mask = (daily_ret.index > start) & (daily_ret.index <= end)
        period = daily_ret.loc[mask, tickers]

        for idx, (date, row) in enumerate(period.iterrows()):
            r = row.mean()
            if idx == 0:
                r -= cost
            port_returns.append({"date": date, "return": r})

        prev_tickers = tickers

    port_df = pd.DataFrame(port_returns).set_index("date")
    total = (1 + port_df["return"]).prod() - 1
    n_years = len(port_df) / 252
    annual = (1 + total) ** (1 / n_years) - 1
    vol = port_df["return"].std() * np.sqrt(252)
    sharpe = (annual - 0.04) / vol
    cum = (1 + port_df["return"]).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    max_dd = dd.min()

    return {
        "total_return": total,
        "annual_return": annual,
        "volatility": vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd
    }

print("Funções definidas.")

### 7.1 Sensibilidade aos hiperparâmetros do XGBoost
Testamos variações de max_depth e learning_rate.
Cada combinação roda o walk-forward inteiro.

In [ ]:
base_params = {
    "n_estimators": 200,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "random_state": 42,
    "n_jobs": -1
}

depth_results = []
for depth in [3, 5, 7, 9]:
    params = {**base_params, "max_depth": depth}
    print(f"Rodando max_depth={depth}...")
    preds = run_walkforward(params)
    metrics = evaluate_predictions(preds)
    metrics["max_depth"] = depth
    depth_results.append(metrics)

depth_df = pd.DataFrame(depth_results)
print("\nResultados por max_depth:")
print(depth_df[["max_depth", "spearman_mean", "spread_q5_q1", "spearman_positive_pct"]].round(4).to_string(index=False))

In [ ]:
lr_results = []
for lr in [0.01, 0.03, 0.05, 0.1, 0.2]:
    params = {**base_params, "learning_rate": lr}
    print(f"Rodando learning_rate={lr}...")
    preds = run_walkforward(params)
    metrics = evaluate_predictions(preds)
    metrics["learning_rate"] = lr
    lr_results.append(metrics)

lr_df = pd.DataFrame(lr_results)
print("\nResultados por learning_rate:")
print(lr_df[["learning_rate", "spearman_mean", "spread_q5_q1", "spearman_positive_pct"]].round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([str(d) for d in depth_df["max_depth"]], depth_df["spread_q5_q1"], color="steelblue")
axes[0].set_xlabel("max_depth")
axes[0].set_ylabel("Spread Q5 - Q1")
axes[0].set_title("Sensibilidade ao max_depth")

axes[1].bar([str(lr) for lr in lr_df["learning_rate"]], lr_df["spread_q5_q1"], color="coral")
axes[1].set_xlabel("learning_rate")
axes[1].set_ylabel("Spread Q5 - Q1")
axes[1].set_title("Sensibilidade ao learning_rate")

plt.tight_layout()
plt.savefig("../results/sensitivity_xgb_params.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Sensibilidade ao custo de transação
Testamos se a estratégia continua viável com custos mais altos.

In [ ]:
print("Rodando modelo base para simulação de portfólio...")
base_preds = run_walkforward(base_params)

cost_results = []
for cost in [0.0, 0.001, 0.002, 0.005, 0.01]:
    metrics = simulate_portfolio(base_preds, transaction_cost=cost)
    metrics["transaction_cost"] = cost
    cost_results.append(metrics)
    print(f"Custo={cost:.1%}: Sharpe={metrics['sharpe']:.2f}, "
          f"Retorno anual={metrics['annual_return']:.2%}")

cost_df = pd.DataFrame(cost_results)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(cost_df["transaction_cost"] * 100, cost_df["sharpe"], "bo-", linewidth=2)
ax1.set_xlabel("Custo de transação (%)")
ax1.set_ylabel("Sharpe Ratio")
ax1.set_title("Sharpe vs custo de transação")
ax1.axhline(y=0, color="red", linestyle="--", alpha=0.5)
ax1.grid(alpha=0.3)

ax2.plot(cost_df["transaction_cost"] * 100, cost_df["annual_return"] * 100, "go-", linewidth=2)
ax2.set_xlabel("Custo de transação (%)")
ax2.set_ylabel("Retorno anual (%)")
ax2.set_title("Retorno anual vs custo de transação")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/sensitivity_transaction_cost.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Sensibilidade à concentração do portfólio
Testamos selecionar top 10%, 20%, 30% e 40% das ações.

In [ ]:
concentration_results = []
for pct in [0.1, 0.15, 0.2, 0.3, 0.4]:
    metrics = simulate_portfolio(base_preds, top_pct=pct)
    metrics["top_pct"] = pct
    concentration_results.append(metrics)
    n_approx = int(150 * pct)
    print(f"Top {pct:.0%} (~{n_approx} ações): Sharpe={metrics['sharpe']:.2f}, "
          f"Retorno anual={metrics['annual_return']:.2%}, "
          f"Max DD={metrics['max_drawdown']:.2%}")

conc_df = pd.DataFrame(concentration_results)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

labels = [f"Top {int(p*100)}%" for p in conc_df["top_pct"]]

ax1.bar(labels, conc_df["sharpe"], color="steelblue")
ax1.set_ylabel("Sharpe Ratio")
ax1.set_title("Sharpe vs concentração do portfólio")

ax2.bar(labels, conc_df["max_drawdown"] * 100, color="salmon")
ax2.set_ylabel("Max Drawdown (%)")
ax2.set_title("Max Drawdown vs concentração")

plt.tight_layout()
plt.savefig("../results/sensitivity_concentration.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.4 Resumo da análise de sensibilidade

In [ ]:
print("=" * 60)
print("RESUMO DA ANÁLISE DE SENSIBILIDADE")
print("=" * 60)

print("\n1. Hiperparâmetros do XGBoost (max_depth):")
print(f"   Spread Q5-Q1 varia de {depth_df['spread_q5_q1'].min():.4f} "
      f"a {depth_df['spread_q5_q1'].max():.4f}")
best_depth = depth_df.loc[depth_df["spread_q5_q1"].idxmax(), "max_depth"]
print(f"   Melhor max_depth: {best_depth}")
stable_depth = depth_df["spread_q5_q1"].std() / depth_df["spread_q5_q1"].mean()
print(f"   Coeficiente de variação: {stable_depth:.2%} ", end="")
print("(ESTÁVEL)" if stable_depth < 0.3 else "(INSTÁVEL)")

print("\n2. Hiperparâmetros do XGBoost (learning_rate):")
print(f"   Spread Q5-Q1 varia de {lr_df['spread_q5_q1'].min():.4f} "
      f"a {lr_df['spread_q5_q1'].max():.4f}")
best_lr = lr_df.loc[lr_df["spread_q5_q1"].idxmax(), "learning_rate"]
print(f"   Melhor learning_rate: {best_lr}")
stable_lr = lr_df["spread_q5_q1"].std() / lr_df["spread_q5_q1"].mean()
print(f"   Coeficiente de variação: {stable_lr:.2%} ", end="")
print("(ESTÁVEL)" if stable_lr < 0.3 else "(INSTÁVEL)")

print("\n3. Custo de transação:")
breakeven = cost_df[cost_df["sharpe"] <= 0]
if len(breakeven) > 0:
    print(f"   A estratégia para de ser lucrativa a partir de {breakeven.iloc[0]['transaction_cost']:.1%}")
else:
    print(f"   A estratégia permanece lucrativa até {cost_df['transaction_cost'].max():.1%} de custo")
print(f"   Sharpe com custo 0: {cost_df.iloc[0]['sharpe']:.2f}")
print(f"   Sharpe com custo 1%: {cost_df.iloc[-1]['sharpe']:.2f}")

print("\n4. Concentração do portfólio:")
best_conc = conc_df.loc[conc_df["sharpe"].idxmax()]
print(f"   Melhor Sharpe: Top {best_conc['top_pct']:.0%} "
      f"(Sharpe={best_conc['sharpe']:.2f})")
print(f"   Mais conservador: Top {conc_df.loc[conc_df['max_drawdown'].idxmax(), 'top_pct']:.0%} "
      f"(menor drawdown)")

print("\n" + "=" * 60)

In [ ]:
results = {
    "xgb_depth": depth_df.to_dict(orient="records"),
    "xgb_lr": lr_df.to_dict(orient="records"),
    "transaction_cost": cost_df.to_dict(orient="records"),
    "concentration": conc_df.to_dict(orient="records")
}

pd.DataFrame(results).to_json("../results/sensitivity_results.json", indent=2)
print("Resultados salvos em results/sensitivity_results.json")